In [0]:
%run ./01_logging_utils.py

In [0]:
from datetime import datetime

view_start = datetime.now()
view_id = new_batch_id("bi_views")


In [0]:

# Creating analytics ready views for Power BI
views = {
    "vw_market_overview": f"""
        SELECT
            d.full_date,
            c.coin_id,
            c.coin_name,
            c.is_stablecoin,
            f.price,
            f.market_cap,
            f.total_volume,
            f.hourly_return_pct,
            f.volume_change_pct,
            f.is_volume_spike,
            f.volume_to_mcap_ratio
        FROM {T_FACT_HOURLY} f
        JOIN {T_DIM_COIN} c ON f.coin_key = c.coin_key
        JOIN {T_DIM_DATE} d ON f.date_key = d.date_key
    """,

    "vw_coin_daily_performance": f"""
        SELECT
            d.full_date,
            c.coin_id,
            c.coin_name,
            c.is_stablecoin,
            f.open_price,
            f.close_price,
            f.high_price,
            f.low_price,
            f.avg_price,
            f.avg_market_cap,
            f.avg_volume,
            f.daily_return_pct,
            f.intraday_range_pct,
            f.volatility_pct,
            f.hours_count
        FROM {T_FACT_DAILY} f
        JOIN {T_DIM_COIN} c ON f.coin_key = c.coin_key
        JOIN {T_DIM_DATE} d ON f.date_key = d.date_key
    """,

    "vw_portfolio_scenarios": f"""
        SELECT
            d.full_date,
            s.scenario_name,
            s.description,
            f.portfolio_return_pct,
            f.market_condition,
            f.is_high_volatility
        FROM {T_FACT_PORTF} f
        JOIN {T_DIM_SCENARIO} s
            ON f.scenario_key = s.scenario_key
        JOIN {T_DIM_DATE} d
            ON f.date_key = d.date_key
    """,

    "vw_scenario_composition": f"""
        SELECT
            s.scenario_name,
            s.description,
            c.coin_id,
            c.coin_name,
            c.is_stablecoin,
            b.weight
        FROM {T_BRIDGE} b
        JOIN {T_DIM_SCENARIO} s
            ON b.scenario_key = s.scenario_key
        JOIN {T_DIM_COIN} c
            ON b.coin_key = c.coin_key
    """,

    "vw_intraday_analysis": f"""
        SELECT
            d.full_date,
            h.hour_key,
            h.hour_label,
            h.day_part,
            c.coin_id,
            c.coin_name,
            f.price,
            f.total_volume,
            f.hourly_return_pct,
            f.volume_change_pct,
            f.is_volume_spike
        FROM {T_FACT_HOURLY} f
        JOIN {T_DIM_COIN} c ON f.coin_key = c.coin_key
        JOIN {T_DIM_DATE} d ON f.date_key = d.date_key
        JOIN {T_DIM_HOUR} h ON f.hour_key = h.hour_key
    """
}

try:
    for view_name, query in views.items():
        spark.sql(
            f"CREATE OR REPLACE VIEW crypto.gold.{view_name} AS {query}"
        )
        print(f"Created view: crypto.gold.{view_name}")

    log_run(
        view_id,
        "bi_views",
        f"Created {len(views)} Power BI views in crypto.gold",
        view_start,
        datetime.now(),
        "SUCCESS"
    )

except Exception as e:
    log_run(
        view_id,
        "bi_views",
        "Creating Power BI analytics views",
        view_start,
        datetime.now(),
        "FAILED",
        err=str(e)[:500]
    )
    raise

# Verifying that all views exist and return data
for view_name in views:
    print(f"\nChecking {view_name}")
    display(spark.sql(
        f"SELECT * FROM crypto.gold.{view_name} LIMIT 5"
    ))

print(f"\nSuccessfully created {len(views)} Power BI views.")
